In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import pandas as pd
import numpy as np
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")
MODELS_DIR = PROJECT_ROOT / "models"
TRAINING_DATA_DIR = PROJECT_ROOT / "training_data"

print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("MODELS_DIR exists:", MODELS_DIR.exists())
print("TRAINING_DATA_DIR exists:", TRAINING_DATA_DIR.exists())

Mounted at /content/drive
PROJECT_ROOT exists: True
MODELS_DIR exists: True
TRAINING_DATA_DIR exists: True


In [2]:
model_folders = [p for p in MODELS_DIR.iterdir() if p.is_dir()]

print("Number of model folders:", len(model_folders))

for p in sorted(model_folders):
    print(p.name)

Number of model folders: 10
final_finetuned_minilm_comment_encoder
ft_comments_only_pro_20260420_142719
ft_post_aware_v2_20260420_172806
ft_post_aware_v3_20260420_174740
ft_post_aware_v4_20260420_182623
ft_v5_3stage_20260420_214501
v6a_fullft_paraphrase_multilingual_minilm_l12_stage1
v6b_fullft_paraphrase_multilingual_minilm_l12_stage1
v7_fullft_minilm_raw_largepairs_1p5M_seq256
v8_baseline_minilm_largepairs_epochs10_es3_lr1e5


In [3]:
important_files = []

patterns = [
    "training_metadata.json",
    "training_summary.csv",
    "trainer_state.json",
    "training_args.bin",
    "*evaluation*.csv",
    "*metrics*.csv",
    "*summary*.json",
    "*summary*.csv",
]

for pattern in patterns:
    for p in MODELS_DIR.rglob(pattern):
        important_files.append(p)

important_files = sorted(set(important_files))

print("Important files found:", len(important_files))

for p in important_files:
    print(p)

Important files found: 25
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_comments_only_pro_20260420_142719/training_metadata.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_post_aware_v2_20260420_172806/training_metadata.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_post_aware_v3_20260420_174740/training_metadata.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_post_aware_v4_20260420_182623/training_metadata.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/v6a_fullft_paraphrase_multilingual_minilm_l12_stage1/eval/Information-Retrieval_evaluation_val_ir_stage1_pairs_results.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/models/v6a_fullft_paraphrase_multilingual_minilm_l12_stage1/training_metadata.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/v6a_fullft_paraphrase_multilingual_m

In [4]:
rows = []

for model_dir in sorted([p for p in MODELS_DIR.iterdir() if p.is_dir()]):
    metadata_path = model_dir / "training_metadata.json"
    summary_path = model_dir / "training_summary.csv"

    row = {
        "model_folder": model_dir.name,
        "model_path": str(model_dir),
        "has_training_metadata": metadata_path.exists(),
        "has_training_summary": summary_path.exists(),
    }

    if metadata_path.exists():
        try:
            with open(metadata_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

            for key in [
                "experiment_name",
                "display_name",
                "base_model",
                "training_type",
                "loss",
                "train_pairs",
                "val_pairs",
                "eval_pairs",
                "train_pairs_path",
                "val_pairs_path",
                "max_seq_length",
                "num_train_epochs",
                "epochs",
                "epochs_max",
                "global_steps",
                "per_device_train_batch_size",
                "per_device_eval_batch_size",
                "batch_size",
                "learning_rate",
                "warmup_ratio",
                "warmup_steps",
                "weight_decay",
                "bf16",
                "fp16",
                "random_state",
                "seed",
                "batch_sampler",
                "early_stopping_patience",
                "final_model_dir",
                "total_parameters",
                "trainable_parameters",
                "trainable_ratio_percent",
                "average_train_loss_full_run",
                "train_runtime_seconds",
                "train_samples_per_second",
                "train_steps_per_second",
            ]:
                row[key] = meta.get(key, None)

        except Exception as e:
            row["metadata_error"] = str(e)

    if summary_path.exists():
        try:
            summary_df = pd.read_csv(summary_path)
            for col in summary_df.columns:
                row[f"summary_{col}"] = summary_df.iloc[0][col]
        except Exception as e:
            row["summary_error"] = str(e)

    rows.append(row)

metadata_audit_df = pd.DataFrame(rows)

display(metadata_audit_df)

OUT_PATH = PROJECT_ROOT / "models_metadata_audit.csv"
metadata_audit_df.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)

,model_folder,model_path,has_training_metadata,has_training_summary,experiment_name,display_name,base_model,training_type,loss,train_pairs,...,summary_Trainable Ratio,summary_Positive Cosine Mean,summary_Positive Cosine Median,summary_Positive Cosine Min,summary_Positive Cosine Max,summary_Random Cosine Mean,summary_Random Cosine Median,summary_Random Cosine Min,summary_Random Cosine Max,summary_Positive-Random Mean Gap
0,final_finetuned_minilm_comment_encoder,/content/drive/MyDrive/semester project/news_c...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ft_comments_only_pro_20260420_142719,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ft_post_aware_v2_20260420_172806,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ft_post_aware_v3_20260420_174740,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ft_post_aware_v4_20260420_182623,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ft_v5_3stage_20260420_214501,/content/drive/MyDrive/semester project/news_c...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,v6a_fullft_paraphrase_multilingual_minilm_l12_...,/content/drive/MyDrive/semester project/news_c...,True,True,v6a_fullft_paraphrase_multilingual_minilm_l12_...,NaN,sentence-transformers/paraphrase-multilingual-...,None,None,NaN,...,100.0,0.4808,0.4689,0.3496,0.7161,0.0674,0.0548,-0.2145,0.6158,0.4134
7,v6b_fullft_paraphrase_multilingual_minilm_l12_...,/content/drive/MyDrive/semester project/news_c...,True,True,v6b_fullft_paraphrase_multilingual_minilm_l12_...,NaN,sentence-transformers/paraphrase-multilingual-...,None,None,NaN,...,100.0,0.4812,0.4670,0.3499,0.7250,0.0698,0.0546,-0.2154,0.6167,0.4114
8,v7_fullft_minilm_raw_largepairs_1p5M_seq256,/content/drive/MyDrive/semester project/news_c...,True,False,v7_fullft_minilm_raw_largepairs_1p5M_seq256,NaN,sentence-transformers/paraphrase-multilingual-...,full_fine_tuning,MultipleNegativesRankingLoss,1500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,/content/drive/MyDrive/semester project/news_c...,True,False,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,NaN,sentence-transformers/paraphrase-multilingual-...,full_fine_tuning_from_baseline,MultipleNegativesRankingLoss,1500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/models_metadata_audit.csv


In [5]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")
MODELS_DIR = PROJECT_ROOT / "models"

rows = []

for trainer_state_path in MODELS_DIR.rglob("trainer_state.json"):
    with open(trainer_state_path, "r", encoding="utf-8") as f:
        state = json.load(f)

    parts = trainer_state_path.parts
    model_folder = None

    if "models" in parts:
        idx = parts.index("models")
        if idx + 1 < len(parts):
            model_folder = parts[idx + 1]

    row = {
        "model_folder": model_folder,
        "checkpoint": trainer_state_path.parent.name,
        "trainer_state_path": str(trainer_state_path),
        "global_step": state.get("global_step"),
        "best_metric": state.get("best_metric"),
        "best_model_checkpoint": state.get("best_model_checkpoint"),
    }

    log_history = state.get("log_history", [])
    eval_logs = [x for x in log_history if "eval_loss" in x]
    train_logs = [x for x in log_history if "loss" in x]

    if train_logs:
        row["last_train_step"] = train_logs[-1].get("step")
        row["last_train_loss"] = train_logs[-1].get("loss")
        row["min_logged_train_loss"] = min(
            [x["loss"] for x in train_logs if x.get("loss") is not None]
        )

    if eval_logs:
        best_eval = min(
            eval_logs,
            key=lambda x: x.get("eval_loss", float("inf"))
        )
        last_eval = eval_logs[-1]

        row["best_eval_step"] = best_eval.get("step")
        row["best_eval_loss"] = best_eval.get("eval_loss")
        row["last_eval_step"] = last_eval.get("step")
        row["last_eval_loss"] = last_eval.get("eval_loss")

        for key, value in best_eval.items():
            if any(m in key.lower() for m in ["accuracy", "mrr", "map", "ndcg"]):
                row[f"best_{key}"] = value

    rows.append(row)

trainer_state_summary = pd.DataFrame(rows)

display(trainer_state_summary)

OUT = PROJECT_ROOT / "trainer_state_summary_extracted.csv"
trainer_state_summary.to_csv(OUT, index=False)

print("Saved:", OUT)

,model_folder,checkpoint,trainer_state_path,global_step,best_metric,best_model_checkpoint,last_train_step,last_train_loss,min_logged_train_loss,best_eval_step,...,best_eval_val_ir_v7_raw_largepairs_cosine_map@100,best_eval_val_ir_v7_raw_largepairs_cosine_mrr@10,best_eval_val_ir_v7_raw_largepairs_cosine_ndcg@10,best_eval_val_ir_v8_cosine_accuracy@1,best_eval_val_ir_v8_cosine_accuracy@10,best_eval_val_ir_v8_cosine_accuracy@3,best_eval_val_ir_v8_cosine_accuracy@5,best_eval_val_ir_v8_cosine_map@100,best_eval_val_ir_v8_cosine_mrr@10,best_eval_val_ir_v8_cosine_ndcg@10
0,v7_fullft_minilm_raw_largepairs_1p5M_seq256,checkpoint-4000,/content/drive/MyDrive/semester project/news_c...,4000,NaN,None,4000,1.570705,1.563786,4000,...,0.275758,0.262221,0.323913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,v7_fullft_minilm_raw_largepairs_1p5M_seq256,checkpoint-5860,/content/drive/MyDrive/semester project/news_c...,5860,NaN,None,5800,1.535031,1.535031,4000,...,0.275758,0.262221,0.323913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,checkpoint-16000,/content/drive/MyDrive/semester project/news_c...,16000,1.615238,/content/drive/MyDrive/semester project/news_c...,16000,1.351128,1.343122,16000,...,NaN,NaN,NaN,0.1622,0.533,0.3174,0.4062,0.2787,0.265407,0.32856
3,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,checkpoint-20000,/content/drive/MyDrive/semester project/news_c...,20000,1.615238,/content/drive/MyDrive/semester project/news_c...,20000,1.291134,1.281221,16000,...,NaN,NaN,NaN,0.1622,0.533,0.3174,0.4062,0.2787,0.265407,0.32856
4,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,checkpoint-22000,/content/drive/MyDrive/semester project/news_c...,22000,1.615238,/content/drive/MyDrive/semester project/news_c...,22000,1.262133,1.262133,16000,...,NaN,NaN,NaN,0.1622,0.533,0.3174,0.4062,0.2787,0.265407,0.32856


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/trainer_state_summary_extracted.csv


In [9]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

V7_STATE = (
    PROJECT_ROOT
    / "models/v7_fullft_minilm_raw_largepairs_1p5M_seq256"
    / "checkpoint-5860"
    / "trainer_state.json"
)

V8_STATE = (
    PROJECT_ROOT
    / "models/v8_baseline_minilm_largepairs_epochs10_es3_lr1e5"
    / "checkpoint-22000"
    / "trainer_state.json"
)

def extract_best_eval_from_state(path, model_name):
    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)

    logs = state.get("log_history", [])

    eval_logs = [x for x in logs if "eval_loss" in x]
    train_logs = [x for x in logs if "loss" in x]

    if not eval_logs:
        raise ValueError(f"No eval logs found for {model_name}")

    # أفضل تقييم حسب eval_loss
    best_eval = min(eval_logs, key=lambda x: x["eval_loss"])

    # آخر train loss قبل أو عند best eval step
    best_step = best_eval["step"]
    train_before_best = [
        x for x in train_logs
        if x.get("step", -1) <= best_step and "loss" in x
    ]

    last_train_before_best = train_before_best[-1] if train_before_best else {}

    row = {
        "Model": model_name,
        "Global Step": state.get("global_step"),
        "Best Model Checkpoint": state.get("best_model_checkpoint"),
        "Best Eval Step": best_eval.get("step"),
        "Train Loss at/Before Best Eval": last_train_before_best.get("loss"),
        "Best Eval Loss": best_eval.get("eval_loss"),
    }

    # استخرج كل مقاييس التقييم كما هي
    for k, v in best_eval.items():
        kl = k.lower()
        if any(m in kl for m in ["accuracy@", "precision@", "recall@", "ndcg@", "mrr@", "map@"]):
            clean = k

            # تنظيف اسم العمود
            clean = clean.replace("eval_val_ir_v7_raw_largepairs_cosine_", "")
            clean = clean.replace("eval_val_ir_v8_cosine_", "")

            row[clean] = v

    return row

v7_row = extract_best_eval_from_state(V7_STATE, "V7 Final")
v8_row = extract_best_eval_from_state(V8_STATE, "V8 Baseline Long Training")

training_metrics_fixed = pd.DataFrame([v7_row, v8_row])

display(training_metrics_fixed.T)

OUT = PROJECT_ROOT / "v7_v8_training_metrics_final_clean.csv"
training_metrics_fixed.to_csv(OUT, index=False)

print("Saved:", OUT)

,0,1
Model,V7 Final,V8 Baseline Long Training
Global Step,5860,22000
Best Model Checkpoint,None,/content/drive/MyDrive/semester project/news_c...
Best Eval Step,4000,16000
Train Loss at/Before Best Eval,1.570705,1.351128
Best Eval Loss,1.649383,1.615238
accuracy@1,0.16,0.1622
accuracy@10,0.523,0.533
accuracy@3,0.311,0.3174
accuracy@5,0.4044,0.4062


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/v7_v8_training_metrics_final_clean.csv


In [10]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")
MODELS_DIR = PROJECT_ROOT / "models"

metadata_files = [
    MODELS_DIR / "ft_comments_only_pro_20260420_142719/training_metadata.json",
    MODELS_DIR / "ft_post_aware_v2_20260420_172806/training_metadata.json",
    MODELS_DIR / "ft_post_aware_v3_20260420_174740/training_metadata.json",
    MODELS_DIR / "ft_post_aware_v4_20260420_182623/training_metadata.json",
    MODELS_DIR / "v6a_fullft_paraphrase_multilingual_minilm_l12_stage1/training_metadata.json",
    MODELS_DIR / "v6b_fullft_paraphrase_multilingual_minilm_l12_stage1/training_metadata.json",
    MODELS_DIR / "v7_fullft_minilm_raw_largepairs_1p5M_seq256/training_metadata.json",
    MODELS_DIR / "v8_baseline_minilm_largepairs_epochs10_es3_lr1e5/training_metadata.json",
]

for path in metadata_files:
    print("=" * 120)
    print(path)
    print("exists:", path.exists())

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("keys:")
        print(list(data.keys()))

        print("\ncontent preview:")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_comments_only_pro_20260420_142719/training_metadata.json
exists: True
keys:
['timestamp', 'base_model_name', 'train_cfg', 'data_counts', 'output_dir', 'embedding_shape_example']

content preview:
{
  "timestamp": "20260420_142719",
  "base_model_name": "paraphrase-multilingual-MiniLM-L12-v2",
  "train_cfg": {
    "min_comments_per_post": 3,
    "candidate_sample_size": 50000,
    "max_train_texts": 35000,
    "epochs": 1,
    "batch_size": 128,
    "max_seq_length": 96,
    "learning_rate": 2e-05,
    "random_state": 42
  },
  "data_counts": {
    "final_unique_train_texts": 35000
  },
  "output_dir": "/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_comments_only_pro_20260420_142719",
  "embedding_shape_example": [
    1,
    384
  ]
}
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_post_aware_v2_20260420_172806/training_metadata.json
exists: True
keys:
['

In [11]:
rows = []

for path in metadata_files:
    model_name = path.parent.name

    row = {
        "model_folder": model_name,
        "metadata_exists": path.exists(),
    }

    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        # نحاول التقاط أشهر الأسماء المحتملة
        field_map = {
            "experiment_name": ["experiment_name", "run_name", "model_name"],
            "base_model": ["base_model", "base_model_name", "model_id"],
            "training_type": ["training_type", "mode", "stage"],
            "loss": ["loss", "loss_function", "objective"],
            "train_examples": ["train_examples", "train_pairs", "train_size", "training_examples", "num_train_examples"],
            "val_examples": ["val_examples", "val_pairs", "val_size", "validation_examples", "num_val_examples"],
            "epochs": ["epochs", "num_train_epochs", "stage1_epochs", "stage2_epochs", "stage3_epochs"],
            "batch_size": ["batch_size", "train_batch_size", "per_device_train_batch_size", "stage1_batch_size"],
            "learning_rate": ["learning_rate", "lr", "stage1_lr", "stage2_lr", "stage3_lr"],
            "max_seq_length": ["max_seq_length", "max_sequence_length"],
            "seed": ["seed", "random_state"],
            "weight_decay": ["weight_decay"],
            "warmup_ratio": ["warmup_ratio"],
            "fp16": ["fp16"],
            "bf16": ["bf16"],
            "trainable_ratio": ["trainable_ratio", "trainable_ratio_percent"],
            "output_dir": ["output_dir", "final_model_dir"],
        }

        for out_key, candidates in field_map.items():
            value = None
            for c in candidates:
                if c in meta:
                    value = meta[c]
                    break
            row[out_key] = value

        # احتفظ بالمفاتيح الأصلية للمراجعة
        row["available_keys"] = ", ".join(meta.keys())

    rows.append(row)

old_models_summary = pd.DataFrame(rows)
display(old_models_summary)

OUT = PROJECT_ROOT / "old_models_metadata_summary.csv"
old_models_summary.to_csv(OUT, index=False)

print("Saved:", OUT)

,model_folder,metadata_exists,experiment_name,base_model,training_type,loss,train_examples,val_examples,epochs,batch_size,learning_rate,max_seq_length,seed,weight_decay,warmup_ratio,fp16,bf16,trainable_ratio,output_dir,available_keys
0,ft_comments_only_pro_20260420_142719,True,None,paraphrase-multilingual-MiniLM-L12-v2,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,/content/drive/MyDrive/semester project/news_c...,"timestamp, base_model_name, train_cfg, data_co..."
1,ft_post_aware_v2_20260420_172806,True,None,paraphrase-multilingual-MiniLM-L12-v2,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,/content/drive/MyDrive/semester project/news_c...,"timestamp, base_model_name, train_cfg_v2, pair..."
2,ft_post_aware_v3_20260420_174740,True,None,paraphrase-multilingual-MiniLM-L12-v2,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,/content/drive/MyDrive/semester project/news_c...,"timestamp, base_model_name, train_cfg_v3, pair..."
3,ft_post_aware_v4_20260420_182623,True,None,paraphrase-multilingual-MiniLM-L12-v2,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,/content/drive/MyDrive/semester project/news_c...,"base_model_name, train_cfg_v4, pairs_dir, pair..."
4,v6a_fullft_paraphrase_multilingual_minilm_l12_...,True,v6a_fullft_paraphrase_multilingual_minilm_l12_...,sentence-transformers/paraphrase-multilingual-...,None,MultipleNegativesRankingLoss,NaN,NaN,1.0,128.0,0.00001,96.0,42.0,0.01,0.1,False,True,100.0,/content/drive/MyDrive/semester project/news_c...,"experiment_name, random_state, base_model, max..."
5,v6b_fullft_paraphrase_multilingual_minilm_l12_...,True,v6b_fullft_paraphrase_multilingual_minilm_l12_...,sentence-transformers/paraphrase-multilingual-...,None,MultipleNegativesRankingLoss,NaN,NaN,1.0,128.0,0.00001,256.0,42.0,0.01,0.1,False,True,100.0,/content/drive/MyDrive/semester project/news_c...,"experiment_name, random_state, base_model, max..."
6,v7_fullft_minilm_raw_largepairs_1p5M_seq256,True,v7_fullft_minilm_raw_largepairs_1p5M_seq256,sentence-transformers/paraphrase-multilingual-...,full_fine_tuning,MultipleNegativesRankingLoss,1500000.0,30000.0,1.0,256.0,0.00001,256.0,42.0,0.01,0.1,False,True,100.0,/content/drive/MyDrive/semester project/news_c...,"experiment_name, base_model, final_model_dir, ..."
7,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,True,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,sentence-transformers/paraphrase-multilingual-...,full_fine_tuning_from_baseline,MultipleNegativesRankingLoss,1500000.0,30000.0,10.0,256.0,0.00001,256.0,42.0,0.01,NaN,False,True,100.0,/content/drive/MyDrive/semester project/news_c...,"experiment_name, base_model, final_model_dir, ..."


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/old_models_metadata_summary.csv


In [12]:
V5_DIR = MODELS_DIR / "ft_v5_3stage_20260420_214501"

print("V5 exists:", V5_DIR.exists())
print("V5 path:", V5_DIR)

for p in sorted(V5_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(V5_DIR))

V5 exists: True
V5 path: /content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501
best_presets_for_supervisor_models.csv
best_presets_for_supervisor_models_rerun.csv
final_model/1_Pooling/config.json
final_model/README.md
final_model/config.json
final_model/config_sentence_transformers.json
final_model/model.safetensors
final_model/modules.json
final_model/sentence_bert_config.json
final_model/tokenizer.json
final_model/tokenizer_config.json
post_context_small_4way_benchmark.csv
practical_50k_best_two_models.csv
practical_50k_full_grid.csv
stage1_model/1_Pooling/config.json
stage1_model/README.md
stage1_model/config.json
stage1_model/config_sentence_transformers.json
stage1_model/model.safetensors
stage1_model/modules.json
stage1_model/sentence_bert_config.json
stage1_model/tokenizer.json
stage1_model/tokenizer_config.json
stage2_model/1_Pooling/config.json
stage2_model/README.md
stage2_model/config.json
stage2_model/config_sentence_transfor

In [13]:
v5_files = []

for p in V5_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in [".csv", ".json", ".txt"]:
        v5_files.append(p)

print("V5 csv/json/txt files:", len(v5_files))

for p in v5_files:
    print(p)

V5 csv/json/txt files: 32
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/v5_checkpoint_benchmark_test_small_large.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/post_context_small_4way_benchmark.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/v5_stage1_test_small_mts_sweep.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/supervisor_fair_comparison_fixed_50k.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/training_metrics_stage1.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/training_metrics_stage2.json
/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/training_metrics_final.json
/content/drive/

In [14]:
for p in v5_files:
    print("=" * 120)
    print(p)

    if p.suffix.lower() == ".csv":
        try:
            temp = pd.read_csv(p, nrows=5)
            print("shape preview:", temp.shape)
            print("columns:", temp.columns.tolist())
            display(temp)
        except Exception as e:
            print("Could not read csv:", e)

    elif p.suffix.lower() == ".json":
        try:
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
            print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
        except Exception as e:
            print("Could not read json:", e)

/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/v5_checkpoint_benchmark_test_small_large.csv
shape preview: (5, 12)
columns: ['test_set', 'model', 'min_topic_size', 'clean_comments', 'valid_topic_comments', 'noise_comments', 'detected_topics', 'noise_ratio_from_clean', 'average_confidence', 'topic_diversity_top10', 'topics_over_time_rows', 'score']


,test_set,model,min_topic_size,clean_comments,valid_topic_comments,noise_comments,detected_topics,noise_ratio_from_clean,average_confidence,topic_diversity_top10,topics_over_time_rows,score
0,test_large,v5_stage1,80,58014,37698,20316,95,35.019133,77.105263,0.852632,263,14641.965137
1,test_large,v4,80,58014,33904,24110,72,41.558934,75.597222,0.904167,204,13231.692791
2,test_large,v5_stage2,80,58014,29943,28071,66,48.386596,76.621212,0.896970,183,11768.835086
3,test_large,v5_final,80,58014,28175,29839,74,51.434137,76.027027,0.656757,246,11085.878199
4,test_small,v4,80,3000,2946,54,2,1.800000,82.500000,1.000000,9,2812.500000


/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/post_context_small_4way_benchmark.csv
shape preview: (4, 14)
columns: ['variant', 'model_key', 'use_post_context', 'context_mode', 'clean_comments', 'valid_topic_comments', 'noise_comments', 'detected_topics', 'noise_ratio_from_clean', 'average_confidence', 'topic_diversity_top10', 'topics_over_time_rows', 'num_context_augmented_comments', 'score']


,variant,model_key,use_post_context,context_mode,clean_comments,valid_topic_comments,noise_comments,detected_topics,noise_ratio_from_clean,average_confidence,topic_diversity_top10,topics_over_time_rows,num_context_augmented_comments,score
0,baseline_plain,baseline,False,never,3000,2907,93,2,3.100000,82.500000,1.000000,9,0,2783.250000
1,v5_stage1_plain,v5_stage1,False,never,3000,1965,1035,8,34.500000,78.625000,0.912500,27,0,2068.750000
2,v5_stage1_ctx_amb,v5_stage1,True,ambiguous_only,3000,1877,1123,9,37.433333,85.000000,0.911111,26,352,2044.861111
3,baseline_ctx_amb,baseline,True,ambiguous_only,3000,1618,1382,7,46.066667,80.714286,0.842857,23,352,1810.071429


/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/v5_stage1_test_small_mts_sweep.csv
shape preview: (5, 11)
columns: ['model', 'min_topic_size', 'clean_comments', 'valid_topic_comments', 'noise_comments', 'detected_topics', 'noise_ratio_from_clean', 'average_confidence', 'topic_diversity_top10', 'topics_over_time_rows', 'score']


...

/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/practical_50k_full_grid.csv
shape preview: (5, 14)
columns: ['Test Set', 'Model', 'min_topic_size', 'n_neighbors', 'min_samples', 'Clean Comments', 'Valid Topic Comments', 'Noise Comments', 'Detected Topics', 'Noise Ratio %', 'Average Confidence', 'Topic Diversity', 'Topics Over Time Rows', 'Score']


,Test Set,Model,min_topic_size,n_neighbors,min_samples,Clean Comments,Valid Topic Comments,Noise Comments,Detected Topics,Noise Ratio %,Average Confidence,Topic Diversity,Topics Over Time Rows,Score
0,fixed_50k_from_test_large,Baseline Plain,60,10,5,50000,30402,19598,23,39.20,76.43,0.94,50,12015.30
1,fixed_50k_from_test_large,Baseline Plain,60,15,10,50000,27924,22076,65,44.15,75.69,0.90,167,11108.04
2,fixed_50k_from_test_large,Baseline Plain,75,15,10,50000,27908,22092,53,44.18,76.40,0.92,143,11107.48
3,fixed_50k_from_test_large,Baseline Plain,80,15,10,50000,27926,22074,51,44.15,77.43,0.91,136,11120.09
4,fixed_50k_from_test_large,V5 Stage 1 Plain,60,15,5,50000,34054,15946,29,31.89,77.17,0.96,72,13411.44


/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/practical_50k_best_two_models.csv
shape preview: (2, 14)
columns: ['Test Set', 'Model', 'min_topic_size', 'n_neighbors', 'min_samples', 'Clean Comments', 'Valid Topic Comments', 'Noise Comments', 'Detected Topics', 'Noise Ratio %', 'Average Confidence', 'Topic Diversity', 'Topics Over Time Rows', 'Score']


,Test Set,Model,min_topic_size,n_neighbors,min_samples,Clean Comments,Valid Topic Comments,Noise Comments,Detected Topics,Noise Ratio %,Average Confidence,Topic Diversity,Topics Over Time Rows,Score
0,fixed_50k_from_test_large,Baseline Plain,60,10,5,50000,30402,19598,23,39.20,76.43,0.94,50,12015.3
1,fixed_50k_from_test_large,V5 Stage 1 Plain,80,15,10,50000,35846,14154,20,28.31,76.80,0.97,57,14044.2


/content/drive/MyDrive/semester project/news_comment_topic_system/models/ft_v5_3stage_20260420_214501/stage1_model/tokenizer.json
{
  "version": "1.0",
  "truncation": {
    "direction": "Right",
    "max_length": 96,
    "strategy": "LongestFirst",
    "stride": 0
  },
  "padding": {
    "strategy": "BatchLongest",
    "direction": "Right",
    "pad_to_multiple_of": null,
    "pad_id": 1,
    "pad_type_id": 0,
    "pad_token": "<pad>"
  },
  "added_tokens": [
    {
      "id": 0,
      "content": "<s>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
      "id": 1,
      "content": "<pad>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
      "id": 2,
      "content": "</s>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
 

In [15]:
all_metric_candidates = []

for p in PROJECT_ROOT.rglob("*.csv"):
    name = p.name.lower()
    path_str = str(p).lower()

    if (
        "metric" in name
        or "result" in name
        or "evaluation" in name
        or "bertopic" in path_str
        or "comparison" in name
        or "practical" in name
        or "benchmark" in name
    ):
        all_metric_candidates.append(p)

print("Metric-like CSV files:", len(all_metric_candidates))

for p in all_metric_candidates:
    print(p)

Metric-like CSV files: 57
/content/drive/MyDrive/semester project/news_comment_topic_system/v7_v8_training_comparison_summary.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/v7_v8_full_training_metrics_comparison.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/v7_v8_full_training_metrics_comparison_fixed.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/v7_v8_training_metrics_final_clean.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/committee_model_audit/02_model_metadata_and_metrics.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/committee_model_audit/06_result_csv_sources.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/committee_model_audit/07_all_evaluation_rows_combined.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/committee_model_audit/18_matched_model_evaluation_rows.csv
/content/drive/MyDrive/semester project/news_comment_top

In [16]:
metric_file_summaries = []

for p in all_metric_candidates:
    try:
        temp = pd.read_csv(p, nrows=3)
        metric_file_summaries.append({
            "file": str(p),
            "columns": ", ".join(temp.columns.tolist()),
        })
    except Exception as e:
        metric_file_summaries.append({
            "file": str(p),
            "columns": f"ERROR: {e}",
        })

metric_file_summaries_df = pd.DataFrame(metric_file_summaries)
display(metric_file_summaries_df)

OUT = PROJECT_ROOT / "metric_file_summaries.csv"
metric_file_summaries_df.to_csv(OUT, index=False)

print("Saved:", OUT)

,file,columns
0,/content/drive/MyDrive/semester project/news_c...,"Model, Best Eval Step, Train Loss, Best Eval L..."
1,/content/drive/MyDrive/semester project/news_c...,"Model, Checkpoint, Global Step, Best Model Che..."
2,/content/drive/MyDrive/semester project/news_c...,"Model, Checkpoint, Global Step, Best Model Che..."
3,/content/drive/MyDrive/semester project/news_c...,"Model, Global Step, Best Model Checkpoint, Bes..."
4,/content/drive/MyDrive/semester project/news_c...,"json_file, path, parent_folder, detected_versi..."
5,/content/drive/MyDrive/semester project/news_c...,"result_file, relative_path, full_path, detecte..."
6,/content/drive/MyDrive/semester project/news_c...,"json_file, path, parent_folder, detected_versi..."
7,/content/drive/MyDrive/semester project/news_c...,"json_file, path, parent_folder, detected_versi..."
8,/content/drive/MyDrive/semester project/news_c...,"json_file, path, parent_folder, detected_versi..."
9,/content/drive/MyDrive/semester project/news_c...,"Test Set, Model, min_topic_size, n_neighbors, ..."


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/metric_file_summaries.csv


In [17]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

v5_final_test_metrics_path = (
    PROJECT_ROOT
    / "bertopic_experiments_clean"
    / "v5_stage1_plain_on_final_test"
    / "v5_stage1_umap15_mcs100_ms10"
    / "metrics_v5_stage1_umap15_mcs100_ms10_on_final_test.csv"
)

print("Exists:", v5_final_test_metrics_path.exists())
print(v5_final_test_metrics_path)

v5_final_test_metrics = pd.read_csv(v5_final_test_metrics_path)
display(v5_final_test_metrics)

Exists: True
/content/drive/MyDrive/semester project/news_comment_topic_system/bertopic_experiments_clean/v5_stage1_plain_on_final_test/v5_stage1_umap15_mcs100_ms10/metrics_v5_stage1_umap15_mcs100_ms10_on_final_test.csv


,Model,Experiment,Total Documents,Valid Topic Comments,Noise Comments,Valid Ratio,Noise Ratio,Detected Topics,Average Confidence All,Average Confidence Valid,...,UMAP n_components,UMAP min_dist,UMAP metric,HDBSCAN min_cluster_size,HDBSCAN min_samples,HDBSCAN metric,Vectorizer ngram_range,Vectorizer min_df,Vectorizer max_df,Model max_seq_length
0,V5 Stage 1 Plain,v5_stage1_umap15_mcs100_ms10_on_final_test,58014,38855,19159,66.98,33.02,111,55.22,82.45,...,5,0.0,cosine,100,10,euclidean,"(1, 2)",2,0.7,96


In [20]:
!pip install -q bertopic umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 1.6 MB/s eta 0:00:00


In [21]:
from bertopic import BERTopic
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

BERTOPIC_MODEL = (
    PROJECT_ROOT
    / "final_bertopic_evaluation_outputs"
    / "final_minilm_comment_encoder_umap15_mcs100_ms10"
    / "final_bertopic_model"
)

print(BERTOPIC_MODEL.exists())

loaded_topic_model = BERTopic.load(str(BERTOPIC_MODEL))
print("Loaded BERTopic model successfully.")

True
Loaded BERTopic model successfully.


In [19]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

V7_FINAL = PROJECT_ROOT / "models/v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model"
ALIAS = PROJECT_ROOT / "models/final_finetuned_minilm_comment_encoder"

print("V7 final exists:", V7_FINAL.exists())
print("Alias exists:", ALIAS.exists())

for name in ["config.json", "model.safetensors", "modules.json", "tokenizer.json"]:
    print(name, (V7_FINAL / name).exists(), (ALIAS / name).exists())

V7 final exists: True
Alias exists: True
config.json True True
model.safetensors True True
modules.json True True
tokenizer.json True True


In [22]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/gdrive/MyDrive/semester project/news_comment_topic_system")

MODELS_DIR = PROJECT_ROOT / "models"
TRAINING_DATA_DIR = PROJECT_ROOT / "training_data"

AUDIT_DIR = PROJECT_ROOT / "final_project_audit_results"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("MODELS_DIR exists:", MODELS_DIR.exists())
print("TRAINING_DATA_DIR exists:", TRAINING_DATA_DIR.exists())
print("AUDIT_DIR:", AUDIT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/semester project/news_comment_topic_system
PROJECT_ROOT exists: True
MODELS_DIR exists: True
TRAINING_DATA_DIR exists: True
AUDIT_DIR: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results


In [23]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_csv(path)


def flatten_dict(d, parent_key="", sep="."):
    items = {}
    if not isinstance(d, dict):
        return items

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        elif isinstance(v, list):
            items[new_key] = json.dumps(v, ensure_ascii=False)
        else:
            items[new_key] = v

    return items


def save_df(df, filename):
    path = AUDIT_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


def show_df(title, df):
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)
    display(df)

In [24]:
artifact_paths = {
    "project_root": PROJECT_ROOT,
    "models_dir": MODELS_DIR,
    "training_data_dir": TRAINING_DATA_DIR,

    "raw_comments": PROJECT_ROOT / "fb_news_comments_1000K_hashed.csv",
    "raw_posts": PROJECT_ROOT / "fb_news_posts_20K.csv",

    "post_split_manifest": TRAINING_DATA_DIR / "v5_post_level_split_20260420_193523/post_split_manifest.csv",
    "comment_split_manifest": TRAINING_DATA_DIR / "v5_post_level_split_20260420_193523/comment_split_manifest_minimal.csv",

    "v5_clean_pools_dir": TRAINING_DATA_DIR / "v5_clean_pools_20260420_194922",
    "test_large_clean_pool": TRAINING_DATA_DIR / "v5_clean_pools_20260420_194922/test_large_clean_pool.csv",

    "v7_pair_mining_dir": TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654",
    "v7_train_pairs_1p5M": TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/train_stage1_pairs_v7_raw_large_balanced_1p5M.csv",
    "v7_val_pairs_30k": TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/val_stage1_pairs_v7_raw_large_balanced_30k.csv",
    "v7_pair_metadata": TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/v7_raw_large_balanced_pair_mining_metadata.json",

    "v7_internal_model": MODELS_DIR / "v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model",
    "final_model_alias": MODELS_DIR / "final_finetuned_minilm_comment_encoder",

    "final_eval_dir": PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10",
    "final_bertopic_model": PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_bertopic_model",
    "final_topic_info": PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_topic_info.csv",
    "final_topic_assignments": PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_topic_assignments.csv",
}

artifact_rows = []

for name, path in artifact_paths.items():
    path = Path(path)
    artifact_rows.append({
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "type": "dir" if path.exists() and path.is_dir() else "file" if path.exists() else "missing"
    })

artifact_df = pd.DataFrame(artifact_rows)

show_df("Artifact Verification", artifact_df)
save_df(artifact_df, "01_artifact_verification.csv")


Artifact Verification


,artifact,path,exists,type
0,project_root,/content/drive/MyDrive/semester project/news_c...,True,dir
1,models_dir,/content/drive/MyDrive/semester project/news_c...,True,dir
2,training_data_dir,/content/drive/MyDrive/semester project/news_c...,True,dir
3,raw_comments,/content/drive/MyDrive/semester project/news_c...,True,file
4,raw_posts,/content/drive/MyDrive/semester project/news_c...,True,file
5,post_split_manifest,/content/drive/MyDrive/semester project/news_c...,True,file
6,comment_split_manifest,/content/drive/MyDrive/semester project/news_c...,True,file
7,v5_clean_pools_dir,/content/drive/MyDrive/semester project/news_c...,True,dir
8,test_large_clean_pool,/content/drive/MyDrive/semester project/news_c...,True,file
9,v7_pair_mining_dir,/content/drive/MyDrive/semester project/news_c...,True,dir


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/01_artifact_verification.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/01_artifact_verification.csv')

In [25]:
pair_metadata_path = TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/v7_raw_large_balanced_pair_mining_metadata.json"
pair_meta = read_json(pair_metadata_path)

if pair_meta is None:
    pair_summary_df = pd.DataFrame([{
        "status": "missing",
        "metadata_path": str(pair_metadata_path)
    }])
else:
    flat_pair_meta = flatten_dict(pair_meta)

    keep_keys = [
        "source",
        "output_dir",
        "train_comments_for_pair_mining",
        "val_comments_for_pair_mining",
        "pair_min_sim",
        "pair_max_sim",
        "max_comments_per_post",
        "max_pairs_per_post_train",
        "max_pairs_per_post_val",
        "train_pairs_before_final_sampling",
        "val_pairs_before_final_sampling",
        "final_train_pairs",
        "final_val_pairs",
        "train_similarity_summary.mean",
        "train_similarity_summary.std",
        "train_similarity_summary.min",
        "train_similarity_summary.25%",
        "train_similarity_summary.50%",
        "train_similarity_summary.75%",
        "train_similarity_summary.max",
        "val_similarity_summary.mean",
        "val_similarity_summary.std",
        "val_similarity_summary.min",
        "val_similarity_summary.25%",
        "val_similarity_summary.50%",
        "val_similarity_summary.75%",
        "val_similarity_summary.max",
    ]

    pair_summary = {
        "metadata_path": str(pair_metadata_path),
        "status": "verified"
    }

    for key in keep_keys:
        pair_summary[key] = flat_pair_meta.get(key, None)

    pair_summary_df = pd.DataFrame([pair_summary])

show_df("Pair Mining Summary", pair_summary_df)
save_df(pair_summary_df, "02_pair_mining_summary.csv")


Pair Mining Summary


,metadata_path,status,source,output_dir,train_comments_for_pair_mining,val_comments_for_pair_mining,pair_min_sim,pair_max_sim,max_comments_per_post,max_pairs_per_post_train,...,train_similarity_summary.50%,train_similarity_summary.75%,train_similarity_summary.max,val_similarity_summary.mean,val_similarity_summary.std,val_similarity_summary.min,val_similarity_summary.25%,val_similarity_summary.50%,val_similarity_summary.75%,val_similarity_summary.max
0,/content/drive/MyDrive/semester project/news_c...,verified,raw_comments,/content/drive/MyDrive/semester project/news_c...,593543,125614,0.45,0.9,200,5000,...,0.594001,0.638951,0.899967,0.607473,0.071215,0.450041,0.563961,0.596498,0.646003,0.899481


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/02_pair_mining_summary.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/02_pair_mining_summary.csv')

In [26]:
model_rows = []

for model_dir in sorted([p for p in MODELS_DIR.iterdir() if p.is_dir()]):
    metadata_path = model_dir / "training_metadata.json"
    summary_path = model_dir / "training_summary.csv"

    row = {
        "model_folder": model_dir.name,
        "model_path": str(model_dir),
        "metadata_exists": metadata_path.exists(),
        "summary_exists": summary_path.exists()
    }

    meta = read_json(metadata_path)

    if meta:
        flat = flatten_dict(meta)

        # Common fields
        possible_keys = {
            "experiment_name": ["experiment_name", "run_name"],
            "display_name": ["display_name"],
            "base_model": ["base_model", "base_model_name"],
            "training_type": ["training_type"],
            "loss": ["loss", "loss_function"],
            "train_pairs": ["train_pairs", "train_pairs_count", "pairs_used", "total_pairs", "data_counts.final_unique_train_texts"],
            "val_pairs": ["val_pairs", "val_pairs_count"],
            "epochs": ["num_train_epochs", "epochs", "epochs_max", "train_cfg.epochs", "train_cfg_v2.epochs", "train_cfg_v3.epochs"],
            "batch_size": ["per_device_train_batch_size", "batch_size", "train_cfg.batch_size", "train_cfg_v2.batch_size", "train_cfg_v3.batch_size", "train_cfg_v4.batch_size"],
            "learning_rate": ["learning_rate", "train_cfg.learning_rate", "train_cfg_v2.learning_rate", "train_cfg_v3.learning_rate", "train_cfg_v4.learning_rate"],
            "max_seq_length": ["max_seq_length", "train_cfg.max_seq_length", "train_cfg_v2.max_seq_length", "train_cfg_v3.max_seq_length", "train_cfg_v4.max_seq_length"],
            "weight_decay": ["weight_decay"],
            "warmup_ratio": ["warmup_ratio", "warmup_steps_ratio"],
            "bf16": ["bf16"],
            "fp16": ["fp16"],
            "random_state": ["random_state", "seed", "train_cfg.random_state", "train_cfg_v2.random_state", "train_cfg_v3.random_state", "train_cfg_v4.random_state"],
            "trainable_ratio_percent": ["trainable_ratio_percent"],
            "final_model_dir": ["final_model_dir"],
            "output_dir": ["output_dir"],
        }

        for out_key, candidates in possible_keys.items():
            value = None
            for c in candidates:
                if c in flat:
                    value = flat[c]
                    break
            row[out_key] = value

        row["available_metadata_keys"] = ", ".join(flat.keys())

    if summary_path.exists():
        try:
            s = pd.read_csv(summary_path)
            for col in s.columns:
                row[f"summary_{col}"] = s.iloc[0][col]
        except Exception as e:
            row["summary_error"] = str(e)

    model_rows.append(row)

model_metadata_df = pd.DataFrame(model_rows)

show_df("Model Metadata Summary", model_metadata_df)
save_df(model_metadata_df, "03_model_metadata_summary.csv")


Model Metadata Summary


,model_folder,model_path,metadata_exists,summary_exists,experiment_name,display_name,base_model,training_type,loss,train_pairs,...,summary_Trainable Ratio,summary_Positive Cosine Mean,summary_Positive Cosine Median,summary_Positive Cosine Min,summary_Positive Cosine Max,summary_Random Cosine Mean,summary_Random Cosine Median,summary_Random Cosine Min,summary_Random Cosine Max,summary_Positive-Random Mean Gap
0,final_finetuned_minilm_comment_encoder,/content/drive/MyDrive/semester project/news_c...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ft_comments_only_pro_20260420_142719,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,paraphrase-multilingual-MiniLM-L12-v2,None,None,35000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ft_post_aware_v2_20260420_172806,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,paraphrase-multilingual-MiniLM-L12-v2,None,None,16000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ft_post_aware_v3_20260420_174740,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,paraphrase-multilingual-MiniLM-L12-v2,None,None,10861.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ft_post_aware_v4_20260420_182623,/content/drive/MyDrive/semester project/news_c...,True,False,None,NaN,paraphrase-multilingual-MiniLM-L12-v2,None,None,4309.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ft_v5_3stage_20260420_214501,/content/drive/MyDrive/semester project/news_c...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,v6a_fullft_paraphrase_multilingual_minilm_l12_...,/content/drive/MyDrive/semester project/news_c...,True,True,v6a_fullft_paraphrase_multilingual_minilm_l12_...,NaN,sentence-transformers/paraphrase-multilingual-...,None,MultipleNegativesRankingLoss,31163.0,...,100.0,0.4808,0.4689,0.3496,0.7161,0.0674,0.0548,-0.2145,0.6158,0.4134
7,v6b_fullft_paraphrase_multilingual_minilm_l12_...,/content/drive/MyDrive/semester project/news_c...,True,True,v6b_fullft_paraphrase_multilingual_minilm_l12_...,NaN,sentence-transformers/paraphrase-multilingual-...,None,MultipleNegativesRankingLoss,31163.0,...,100.0,0.4812,0.4670,0.3499,0.7250,0.0698,0.0546,-0.2154,0.6167,0.4114
8,v7_fullft_minilm_raw_largepairs_1p5M_seq256,/content/drive/MyDrive/semester project/news_c...,True,False,v7_fullft_minilm_raw_largepairs_1p5M_seq256,NaN,sentence-transformers/paraphrase-multilingual-...,full_fine_tuning,MultipleNegativesRankingLoss,1500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,/content/drive/MyDrive/semester project/news_c...,True,False,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,NaN,sentence-transformers/paraphrase-multilingual-...,full_fine_tuning_from_baseline,MultipleNegativesRankingLoss,1500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/03_model_metadata_summary.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/03_model_metadata_summary.csv')

In [27]:
V5_DIR = MODELS_DIR / "ft_v5_3stage_20260420_214501"

v5_metric_files = {
    "stage1": V5_DIR / "training_metrics_stage1.json",
    "stage2": V5_DIR / "training_metrics_stage2.json",
    "final": V5_DIR / "training_metrics_final.json",
}

v5_rows = []

for stage_name, path in v5_metric_files.items():
    meta = read_json(path)

    if meta is None:
        v5_rows.append({
            "stage_file": stage_name,
            "path": str(path),
            "exists": False
        })
        continue

    flat = flatten_dict(meta)

    row = {
        "stage_file": stage_name,
        "path": str(path),
        "exists": True,

        "random_state": flat.get("cfg.random_state"),
        "max_seq_length": flat.get("cfg.max_seq_length"),
        "use_amp": flat.get("cfg.use_amp"),

        "stage1_epochs": flat.get("cfg.stage1_epochs"),
        "stage1_batch_size": flat.get("cfg.stage1_batch_size"),
        "stage1_lr": flat.get("cfg.stage1_lr"),

        "stage2_epochs": flat.get("cfg.stage2_epochs"),
        "stage2_batch_size": flat.get("cfg.stage2_batch_size"),
        "stage2_lr": flat.get("cfg.stage2_lr"),

        "stage3_epochs": flat.get("cfg.stage3_epochs"),
        "stage3_batch_size": flat.get("cfg.stage3_batch_size"),
        "stage3_lr": flat.get("cfg.stage3_lr"),

        "triplet_margin": flat.get("cfg.triplet_margin"),

        "train_stage1": flat.get("dataset_sizes.train_stage1"),
        "train_stage2": flat.get("dataset_sizes.train_stage2"),
        "train_stage3": flat.get("dataset_sizes.train_stage3"),

        "val_stage1": flat.get("dataset_sizes.val_stage1"),
        "val_stage2": flat.get("dataset_sizes.val_stage2"),
        "val_stage3": flat.get("dataset_sizes.val_stage3"),

        "stage1_val_stage1_mean_cosine": flat.get("stage1_eval.val_stage1.mean_cosine"),
        "stage1_val_stage1_median_cosine": flat.get("stage1_eval.val_stage1.median_cosine"),

        "stage2_val_stage1_mean_cosine": flat.get("stage2_eval.val_stage1.mean_cosine"),
        "stage2_val_stage1_median_cosine": flat.get("stage2_eval.val_stage1.median_cosine"),
        "stage2_val_stage2_mean_cosine": flat.get("stage2_eval.val_stage2.mean_cosine"),
        "stage2_val_stage2_median_cosine": flat.get("stage2_eval.val_stage2.median_cosine"),

        "stage3_val_stage1_mean_cosine": flat.get("stage3_eval.val_stage1.mean_cosine"),
        "stage3_val_stage1_median_cosine": flat.get("stage3_eval.val_stage1.median_cosine"),
        "stage3_val_stage2_mean_cosine": flat.get("stage3_eval.val_stage2.mean_cosine"),
        "stage3_val_stage2_median_cosine": flat.get("stage3_eval.val_stage2.median_cosine"),
    }

    v5_rows.append(row)

v5_training_metrics_df = pd.DataFrame(v5_rows)

show_df("V5 Training Metrics Summary", v5_training_metrics_df)
save_df(v5_training_metrics_df, "04_v5_training_metrics_summary.csv")


V5 Training Metrics Summary


,stage_file,path,exists,random_state,max_seq_length,use_amp,stage1_epochs,stage1_batch_size,stage1_lr,stage2_epochs,...,stage1_val_stage1_mean_cosine,stage1_val_stage1_median_cosine,stage2_val_stage1_mean_cosine,stage2_val_stage1_median_cosine,stage2_val_stage2_mean_cosine,stage2_val_stage2_median_cosine,stage3_val_stage1_mean_cosine,stage3_val_stage1_median_cosine,stage3_val_stage2_mean_cosine,stage3_val_stage2_median_cosine
0,stage1,/content/drive/MyDrive/semester project/news_c...,True,42,96,True,1,32,0.00001,1,...,0.502501,0.487613,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,stage2,/content/drive/MyDrive/semester project/news_c...,True,42,96,True,1,32,0.00001,1,...,0.502501,0.487613,0.842579,0.848846,0.6521,0.65583,NaN,NaN,NaN,NaN
2,final,/content/drive/MyDrive/semester project/news_c...,True,42,96,True,1,32,0.00001,1,...,0.502501,0.487613,0.842579,0.848846,0.6521,0.65583,0.997933,0.998887,0.988906,0.991228


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/04_v5_training_metrics_summary.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/04_v5_training_metrics_summary.csv')

In [28]:
trainer_rows = []

trainer_state_paths = list(MODELS_DIR.rglob("trainer_state.json"))

for path in trainer_state_paths:
    state = read_json(path)

    parts = path.parts
    model_folder = None

    if "models" in parts:
        idx = parts.index("models")
        if idx + 1 < len(parts):
            model_folder = parts[idx + 1]

    logs = state.get("log_history", [])
    eval_logs = [x for x in logs if "eval_loss" in x]
    train_logs = [x for x in logs if "loss" in x]

    row = {
        "model_folder": model_folder,
        "checkpoint": path.parent.name,
        "trainer_state_path": str(path),
        "global_step": state.get("global_step"),
        "best_metric": state.get("best_metric"),
        "best_model_checkpoint": state.get("best_model_checkpoint"),
    }

    if train_logs:
        row["last_train_step"] = train_logs[-1].get("step")
        row["last_train_loss"] = train_logs[-1].get("loss")
        row["min_logged_train_loss"] = min([x["loss"] for x in train_logs if "loss" in x])

    if eval_logs:
        best_eval = min(eval_logs, key=lambda x: x.get("eval_loss", float("inf")))
        row["best_eval_step"] = best_eval.get("step")
        row["best_eval_loss"] = best_eval.get("eval_loss")

        for k, v in best_eval.items():
            key_lower = k.lower()
            if any(m in key_lower for m in ["accuracy@", "precision@", "recall@", "mrr@", "ndcg@", "map@"]):
                clean = k.replace("eval_", "")
                row[clean] = v

    trainer_rows.append(row)

trainer_metrics_df = pd.DataFrame(trainer_rows)

show_df("Trainer State Metrics", trainer_metrics_df)
save_df(trainer_metrics_df, "05_trainer_state_metrics.csv")


Trainer State Metrics


,model_folder,checkpoint,trainer_state_path,global_step,best_metric,best_model_checkpoint,last_train_step,last_train_loss,min_logged_train_loss,best_eval_step,...,val_ir_v8_cosine_mrr@10,val_ir_v8_cosine_ndcg@10,val_ir_v8_cosine_precision@1,val_ir_v8_cosine_precision@10,val_ir_v8_cosine_precision@3,val_ir_v8_cosine_precision@5,val_ir_v8_cosine_recall@1,val_ir_v8_cosine_recall@10,val_ir_v8_cosine_recall@3,val_ir_v8_cosine_recall@5
0,v7_fullft_minilm_raw_largepairs_1p5M_seq256,checkpoint-4000,/content/drive/MyDrive/semester project/news_c...,4000,NaN,None,4000,1.570705,1.563786,4000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,v7_fullft_minilm_raw_largepairs_1p5M_seq256,checkpoint-5860,/content/drive/MyDrive/semester project/news_c...,5860,NaN,None,5800,1.535031,1.535031,4000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,checkpoint-16000,/content/drive/MyDrive/semester project/news_c...,16000,1.615238,/content/drive/MyDrive/semester project/news_c...,16000,1.351128,1.343122,16000,...,0.265407,0.32856,0.1622,0.0533,0.1058,0.08124,0.1622,0.533,0.3174,0.4062
3,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,checkpoint-20000,/content/drive/MyDrive/semester project/news_c...,20000,1.615238,/content/drive/MyDrive/semester project/news_c...,20000,1.291134,1.281221,16000,...,0.265407,0.32856,0.1622,0.0533,0.1058,0.08124,0.1622,0.533,0.3174,0.4062
4,v8_baseline_minilm_largepairs_epochs10_es3_lr1e5,checkpoint-22000,/content/drive/MyDrive/semester project/news_c...,22000,1.615238,/content/drive/MyDrive/semester project/news_c...,22000,1.262133,1.262133,16000,...,0.265407,0.32856,0.1622,0.0533,0.1058,0.08124,0.1622,0.533,0.3174,0.4062


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/05_trainer_state_metrics.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/05_trainer_state_metrics.csv')

In [29]:
V7_STATE = MODELS_DIR / "v7_fullft_minilm_raw_largepairs_1p5M_seq256/checkpoint-5860/trainer_state.json"
V8_STATE = MODELS_DIR / "v8_baseline_minilm_largepairs_epochs10_es3_lr1e5/checkpoint-22000/trainer_state.json"

def extract_best_eval_from_state(path, model_name):
    state = read_json(path)
    logs = state.get("log_history", [])

    eval_logs = [x for x in logs if "eval_loss" in x]
    train_logs = [x for x in logs if "loss" in x]

    best_eval = min(eval_logs, key=lambda x: x["eval_loss"])
    best_step = best_eval["step"]

    train_before_best = [
        x for x in train_logs
        if x.get("step", -1) <= best_step and "loss" in x
    ]

    last_train = train_before_best[-1] if train_before_best else {}

    row = {
        "Model": model_name,
        "Global Step": state.get("global_step"),
        "Best Model Checkpoint": state.get("best_model_checkpoint"),
        "Best Eval Step": best_step,
        "Train Loss at Best Eval": last_train.get("loss"),
        "Best Eval Loss": best_eval.get("eval_loss"),
    }

    for k, v in best_eval.items():
        kl = k.lower()
        if any(m in kl for m in ["accuracy@", "precision@", "recall@", "ndcg@", "mrr@", "map@"]):
            clean = k
            clean = clean.replace("eval_val_ir_v7_raw_largepairs_cosine_", "")
            clean = clean.replace("eval_val_ir_v8_cosine_", "")
            row[clean] = v

    return row

v7_row = extract_best_eval_from_state(V7_STATE, "V7 Final")
v8_row = extract_best_eval_from_state(V8_STATE, "V8 Baseline Long Training")

v7_v8_training_comparison_df = pd.DataFrame([v7_row, v8_row])

show_df("V7 vs V8 Full Training Metrics", v7_v8_training_comparison_df.T)
save_df(v7_v8_training_comparison_df, "06_v7_v8_full_training_metrics.csv")


V7 vs V8 Full Training Metrics


,0,1
Model,V7 Final,V8 Baseline Long Training
Global Step,5860,22000
Best Model Checkpoint,None,/content/drive/MyDrive/semester project/news_c...
Best Eval Step,4000,16000
Train Loss at Best Eval,1.570705,1.351128
Best Eval Loss,1.649383,1.615238
accuracy@1,0.16,0.1622
accuracy@10,0.523,0.533
accuracy@3,0.311,0.3174
accuracy@5,0.4044,0.4062


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/06_v7_v8_full_training_metrics.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/06_v7_v8_full_training_metrics.csv')

In [30]:
metric_files = {
    "V5 Controlled": PROJECT_ROOT / "bertopic_experiments_clean/v5_stage1_plain_on_final_test/v5_stage1_umap15_mcs100_ms10/metrics_v5_stage1_umap15_mcs100_ms10_on_final_test.csv",
    "V6-B": PROJECT_ROOT / "bertopic_experiments_clean/v6b_fullft_minilm_stage1_seq256/v6b_seq256_umap15_mcs100_ms10/metrics_v6b_seq256_umap15_mcs100_ms10.csv",
    "V7 Final": PROJECT_ROOT / "bertopic_experiments_clean/v7_fullft_minilm_raw_largepairs_1p5M_seq256/v7_seq256_umap15_mcs100_ms10/metrics_v7_seq256_umap15_mcs100_ms10.csv",
    "V8": PROJECT_ROOT / "models/v8_baseline_minilm_largepairs_epochs10_es3_lr1e5/bertopic_v8_umap15_mcs100_ms10/metrics_v8_umap15_mcs100_ms10.csv",
}

bertopic_rows = []

for label, path in metric_files.items():
    if not path.exists():
        bertopic_rows.append({
            "Label": label,
            "metrics_exists": False,
            "metrics_path": str(path)
        })
        continue

    df_m = pd.read_csv(path)
    row = df_m.iloc[0].to_dict()
    row["Label"] = label
    row["metrics_exists"] = True
    row["metrics_path"] = str(path)

    bertopic_rows.append(row)

bertopic_metrics_df = pd.DataFrame(bertopic_rows)

show_df("BERTopic Metrics Verified Files", bertopic_metrics_df)
save_df(bertopic_metrics_df, "07_bertopic_metrics_verified.csv")


BERTopic Metrics Verified Files


,Model,Experiment,Total Documents,Valid Topic Comments,Noise Comments,Valid Ratio,Noise Ratio,Detected Topics,Average Confidence All,Average Confidence Valid,...,HDBSCAN min_samples,HDBSCAN metric,Vectorizer ngram_range,Vectorizer min_df,Vectorizer max_df,Model max_seq_length,Label,metrics_exists,metrics_path,Embedding Model Path
0,V5 Stage 1 Plain,v5_stage1_umap15_mcs100_ms10_on_final_test,58014.0,38855.0,19159.0,66.98,33.02,111.0,55.22,82.45,...,10.0,euclidean,"(1, 2)",2.0,0.7,96.0,V5 Controlled,True,/content/drive/MyDrive/semester project/news_c...,NaN
1,V6-B FullFT MiniLM Stage1 Seq256,v6b_seq256_umap15_mcs100_ms10,58014.0,37134.0,20880.0,64.01,35.99,104.0,51.73,80.82,...,10.0,euclidean,"(1, 2)",2.0,0.7,256.0,V6-B,True,/content/drive/MyDrive/semester project/news_c...,/content/drive/MyDrive/semester project/news_c...
2,V7 FullFT MiniLM RawLargePairs Seq256,v7_seq256_umap15_mcs100_ms10,58014.0,39458.0,18556.0,68.01,31.99,94.0,56.13,82.52,...,10.0,euclidean,"(1, 2)",2.0,0.7,NaN,V7 Final,True,/content/drive/MyDrive/semester project/news_c...,NaN
3,NaN,NaN,58014.0,37504.0,20510.0,64.65,35.35,110.0,33.46,49.72,...,NaN,NaN,NaN,NaN,NaN,NaN,V8,True,/content/drive/MyDrive/semester project/news_c...,NaN


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/07_bertopic_metrics_verified.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/07_bertopic_metrics_verified.csv')

In [31]:
def get_metric(row, name):
    return row.get(name, np.nan)

rows = []

for _, r in bertopic_metrics_df.iterrows():
    rows.append({
        "Model": r.get("Label"),
        "Total Documents": get_metric(r, "Total Documents"),
        "Valid Topic Comments": get_metric(r, "Valid Topic Comments"),
        "Noise Comments": get_metric(r, "Noise Comments"),
        "Valid Ratio": get_metric(r, "Valid Ratio"),
        "Noise Ratio": get_metric(r, "Noise Ratio"),
        "Detected Topics": get_metric(r, "Detected Topics"),
        "Average Confidence All": get_metric(r, "Average Confidence All"),
        "Average Confidence Valid": get_metric(r, "Average Confidence Valid"),
        "Median Confidence Valid": get_metric(r, "Median Confidence Valid"),
        "Topic Diversity": get_metric(r, "Topic Diversity"),
        "Source File": r.get("metrics_path")
    })

final_comparison_df = pd.DataFrame(rows)

show_df("Final Model Comparison", final_comparison_df)
save_df(final_comparison_df, "08_final_model_comparison.csv")


Final Model Comparison


,Model,Total Documents,Valid Topic Comments,Noise Comments,Valid Ratio,Noise Ratio,Detected Topics,Average Confidence All,Average Confidence Valid,Median Confidence Valid,Topic Diversity,Source File
0,V5 Controlled,58014.0,38855.0,19159.0,66.98,33.02,111.0,55.22,82.45,98.96,0.9036,/content/drive/MyDrive/semester project/news_c...
1,V6-B,58014.0,37134.0,20880.0,64.01,35.99,104.0,51.73,80.82,93.50,0.9048,/content/drive/MyDrive/semester project/news_c...
2,V7 Final,58014.0,39458.0,18556.0,68.01,31.99,94.0,56.13,82.52,98.08,0.9011,/content/drive/MyDrive/semester project/news_c...
3,V8,58014.0,37504.0,20510.0,64.65,35.35,110.0,33.46,49.72,33.82,0.8755,/content/drive/MyDrive/semester project/news_c...


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/08_final_model_comparison.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/08_final_model_comparison.csv')

In [32]:
artifact_rows = [
    {
        "Artifact": "Final V7 internal model",
        "Path": str(MODELS_DIR / "v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model"),
        "Exists": (MODELS_DIR / "v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model").exists()
    },
    {
        "Artifact": "Clean final model alias",
        "Path": str(MODELS_DIR / "final_finetuned_minilm_comment_encoder"),
        "Exists": (MODELS_DIR / "final_finetuned_minilm_comment_encoder").exists()
    },
    {
        "Artifact": "V7 train pairs",
        "Path": str(TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/train_stage1_pairs_v7_raw_large_balanced_1p5M.csv"),
        "Exists": (TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/train_stage1_pairs_v7_raw_large_balanced_1p5M.csv").exists()
    },
    {
        "Artifact": "V7 validation pairs",
        "Path": str(TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/val_stage1_pairs_v7_raw_large_balanced_30k.csv"),
        "Exists": (TRAINING_DATA_DIR / "v7_raw_pair_mining_20260428_232654/val_stage1_pairs_v7_raw_large_balanced_30k.csv").exists()
    },
    {
        "Artifact": "Final BERTopic model",
        "Path": str(PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_bertopic_model"),
        "Exists": (PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_bertopic_model").exists()
    },
    {
        "Artifact": "Final topic info",
        "Path": str(PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_topic_info.csv"),
        "Exists": (PROJECT_ROOT / "final_bertopic_evaluation_outputs/final_minilm_comment_encoder_umap15_mcs100_ms10/final_topic_info.csv").exists()
    },
]

artifact_summary_df = pd.DataFrame(artifact_rows)

show_df("Artifact Summary", artifact_summary_df)
save_df(artifact_summary_df, "09_artifact_summary.csv")


Artifact Summary


,Artifact,Path,Exists
0,Final V7 internal model,/content/drive/MyDrive/semester project/news_c...,True
1,Clean final model alias,/content/drive/MyDrive/semester project/news_c...,True
2,V7 train pairs,/content/drive/MyDrive/semester project/news_c...,True
3,V7 validation pairs,/content/drive/MyDrive/semester project/news_c...,True
4,Final BERTopic model,/content/drive/MyDrive/semester project/news_c...,True
5,Final topic info,/content/drive/MyDrive/semester project/news_c...,True


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/09_artifact_summary.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/09_artifact_summary.csv')

In [33]:
audit_notes = [
    {
        "Topic": "Final adopted model",
        "Conclusion": "V7 / Final Fine-Tuned MiniLM Comment Encoder is the adopted final model."
    },
    {
        "Topic": "V8",
        "Conclusion": "V8 improved training-side retrieval metrics but performed worse than V7 in downstream BERTopic, so it was rejected."
    },
    {
        "Topic": "V5",
        "Conclusion": "V5 controlled metrics are now verified from metrics_v5_stage1_umap15_mcs100_ms10_on_final_test.csv."
    },
    {
        "Topic": "Classification",
        "Conclusion": "The project is unsupervised topic modeling, not supervised classification. Do not use final accuracy/precision/recall as final metrics."
    },
    {
        "Topic": "LLM",
        "Conclusion": "LLM, if used later, should be only for topic labeling/summarization, not for clustering or training."
    },
    {
        "Topic": "Outlier reduction",
        "Conclusion": "reduce_outliers should be optional post-processing, not the main reported result."
    }
]

audit_notes_df = pd.DataFrame(audit_notes)

show_df("Final Audit Notes", audit_notes_df)
save_df(audit_notes_df, "10_final_audit_notes.csv")


Final Audit Notes


,Topic,Conclusion
0,Final adopted model,V7 / Final Fine-Tuned MiniLM Comment Encoder i...
1,V8,V8 improved training-side retrieval metrics bu...
2,V5,V5 controlled metrics are now verified from me...
3,Classification,"The project is unsupervised topic modeling, no..."
4,LLM,"LLM, if used later, should be only for topic l..."
5,Outlier reduction,reduce_outliers should be optional post-proces...


Saved: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/10_final_audit_notes.csv


PosixPath('/content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results/10_final_audit_notes.csv')

In [34]:
from pathlib import Path
import shutil
from google.colab import files

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

AUDIT_DIR = PROJECT_ROOT / "final_project_audit_results"

ZIP_PATH = PROJECT_ROOT / "final_project_audit_results.zip"

# حذف نسخة zip قديمة إن وجدت
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# إنشاء zip
shutil.make_archive(
    base_name=str(ZIP_PATH).replace(".zip", ""),
    format="zip",
    root_dir=str(AUDIT_DIR)
)

print("Created:", ZIP_PATH)
print("Exists:", ZIP_PATH.exists())

# تحميل الملف إلى جهازك
files.download(str(ZIP_PATH))

Created: /content/drive/MyDrive/semester project/news_comment_topic_system/final_project_audit_results.zip
Exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
for p in AUDIT_DIR.iterdir():
    print(p.name)

01_artifact_verification.csv
02_pair_mining_summary.csv
03_model_metadata_summary.csv
04_v5_training_metrics_summary.csv
05_trainer_state_metrics.csv
06_v7_v8_full_training_metrics.csv
07_bertopic_metrics_verified.csv
08_final_model_comparison.csv
09_artifact_summary.csv
10_final_audit_notes.csv
